In [0]:
from pyspark.sql.functions import col, to_timestamp

# Define the path to your Volume
bronze_volume_path = "/Volumes/us-accidents/bronze/raw-data/US_Accidents_March23.csv"

raw_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(bronze_volume_path)

In [0]:
raw_df.printSchema()


In [0]:
from pyspark.sql.functions import (
    col, to_timestamp, hour, when, mean, broadcast, 
    regexp_replace, lower, trim, count
)

catalog = "us-accidents"
volume_path = f"/Volumes/{catalog}/bronze/raw-data/US_Accidents_March23.csv"
silver_table = f"`{catalog}`.silver.accidents_processed"

#READ BRONZE DATA
df = spark.read.format("csv").option("header", "true").load(volume_path)

df = df.withColumn("Start_Time", to_timestamp(col("Start_Time"))) \
       .withColumn("End_Time", to_timestamp(col("End_Time")))


# 6 AM to 5 PM is Day, else its Night
df = df.withColumn("Sunrise_Sunset", 
    when(col("Sunrise_Sunset").isNull(), 
         when((hour(col("Start_Time")) >= 6) & (hour(col("Start_Time")) <= 17), "Day").otherwise("Night")
    ).otherwise(col("Sunrise_Sunset"))
)

# drop waste columns
cols_to_drop = [
    'ID', 'Source', 'Weather_Timestamp', 'Airport_Code', 'Timezone', 'Country', 
    'Description', 'End_Lat', 'End_Lng', 'Wind_Chill(F)', 'Wind_Direction', 
    'Civil_Twilight', 'Humidity(%)', 'Pressure(in)', 'Precipitation(in)', 
    'Nautical_Twilight', 'Astronomical_Twilight'
]
df = df.drop(*cols_to_drop)


weather_stats = df.dropna(subset=['Weather_Condition']).groupBy('Weather_Condition').agg(
    mean(col("Wind_Speed(mph)").cast("float")).alias("avg_wind"),
    mean(col("Visibility(mi)").cast("float")).alias("avg_vis")
)

df = df.join(broadcast(weather_stats), "Weather_Condition", "left") \
       .withColumn("Wind_Speed(mph)", when(col("Wind_Speed(mph)").isNull(), col("avg_wind")).otherwise(col("Wind_Speed(mph)").cast("float"))) \
       .withColumn("Visibility(mi)", when(col("Visibility(mi)").isNull(), col("avg_vis")).otherwise(col("Visibility(mi)").cast("float"))) \
       .drop("avg_wind", "avg_vis")

# filter top 25 weather conditions
top_25 = [r[0] for r in df.groupBy("Weather_Condition").count().orderBy(col("count").desc()).limit(25).collect()]
df = df.filter(col("Weather_Condition").isin(top_25))
df = df.withColumn("Weather_Condition", lower(trim(regexp_replace(col("Weather_Condition"), r'[/\\-]', ' '))))
df = df.withColumn("Weather_Condition", regexp_replace(col("Weather_Condition"), r'\s+', ' '))

df = df.withColumn("Weather_Condition", 
    when(col("Weather_Condition").rlike('light rain with thunder|thunder|t storm'), 'Rainy_Thunder')
    .when(col("Weather_Condition").rlike('light rain|light drizzle|rain'), 'Rain_Drizzle')
    .when(col("Weather_Condition").rlike('light snow|snow'), 'Snow')
    .when(col("Weather_Condition").rlike('fair|clear'), 'Clear_Fair')
    .when(col("Weather_Condition").rlike('mostly cloudy windy|cloudy windy'), 'Cloudy_Windy')
    .when(col("Weather_Condition").rlike('mostly cloudy|cloudy|partly cloudy|overcast|scattered clouds'), 'Cloudy')
    .otherwise(col("Weather_Condition"))
)

# remove outliers
def get_iqr_bounds(column, lower_multiplier=1.5, upper_multiplier=1.5):
    
    quantiles = df.select(col(column).cast("float").alias("temp_col")).approxQuantile("temp_col", [0.25, 0.75], 0.05)
    iqr = quantiles[1] - quantiles[0]
    return quantiles[0] - (lower_multiplier * iqr), quantiles[1] + (upper_multiplier * iqr)

wind_lower, wind_upper = get_iqr_bounds("Wind_Speed(mph)")
temp_lower, temp_upper = get_iqr_bounds("Temperature(F)", lower_multiplier=3.0, upper_multiplier=1.5)

df = df.filter(
    (col("Wind_Speed(mph)") >= wind_lower) & (col("Wind_Speed(mph)") <= wind_upper) &
    (col("Temperature(F)").cast("float") >= temp_lower) & (col("Temperature(F)").cast("float") <= temp_upper)
)

df = df.dropDuplicates()
for c in ["State", "Street", "County", "City"]:
    df = df.withColumn(c, trim(col(c)))

df = df.withColumnRenamed("Distance(mi)", "Distance_mi") \
       .withColumnRenamed("Temperature(F)", "Temperature_F") \
       .withColumnRenamed("Visibility(mi)", "Visibility_mi") \
       .withColumnRenamed("Wind_Speed(mph)", "Wind_Speed_mph")

# save to Silver Schema as a Delta Table
df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

In [0]:
%sql
SELECT * FROM `us-accidents`.silver.accidents_processed LIMIT 1